# Solver LBTE -- Notebook Base (S26)

Implementacion completa del solver BTE para fotones kV con discretizacion S26.

| Celda | Contenido | Referencia |
|:-----:|:----------|:-----------|
| 0 | Imports y backend GPU/CPU | --- |
| 1 | Constantes, grupos de energia, cuadratura S26 | Acuros Sec.2.B; Vassiliev Sec.7.2 |
| 2 | PointCrossSections | Acuros Eq.(1); NJOY GAMINR Sec.9 |
| 3 | UncollidedFluence | Acuros Eq.(3), Eq.(15) |
| 4 | MultiGroupCrossSections | Vassiliev Sec.7.2.1; Acuros Eq.(16) |
| 5 | FirstCollisionSource | Acuros Eq.(4), Eq.(16) |
| 6 | BTE | Fachada principal del solver |
| 7 | Instanciacion y ejemplo minimo | --- |

**Unidades:** energia keV - longitud cm - angulo sr - seccion eficaz cm^-1  
**Backend:** CuPy (GPU) si disponible, NumPy (CPU) en otro caso.


## Celda 0 --- Imports y backend

In [1]:
# CELDA 0 -- Imports y backend GPU/CPU

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# -- Backend GPU (CuPy) / CPU (NumPy)
try:
    import cupy as cp
    _CUPY_OK = True
    xp = cp
    print('OK CuPy disponible -- GPU activada')
    print(f'   CuPy {cp.__version__}  |  GPU mem: {cp.cuda.Device().mem_info[1]/1024**3:.1f} GB')
except Exception as e:
    _CUPY_OK = False
    xp = np
    print(f'WARNING CuPy no disponible: {e} -- usando NumPy')

import spekpy as sp
from raytrace import siddonraytracer
from raytrace.raytrace_ext import raytrace

print(f'\nOK Imports completados  |  NumPy {np.__version__}')


OK CuPy disponible -- GPU activada
   CuPy 13.6.0  |  GPU mem: 4.0 GB

OK Imports completados  |  NumPy 1.26.4


## Celda 1 --- Constantes fisicas, grupos de energia y cuadratura S26

Constantes CGS, malla multigrupo 10x10 keV y las 26 ordenadas discretas con pesos omega_m [sr].

Ref: Acuros Sec.2.B; Vassiliev Sec.7.2


In [2]:
# CELDA 1 -- Constantes fisicas, grupos de energia y cuadratura S26

import math
import numpy as np
import pandas as pd

# -- Constantes fisicas (CGS)
N_A          = 6.02214076e23           # mol^-1
r_e_cm       = 2.8179403262e-13        # cm
sigma_T_cm2  = (8.0*math.pi/3.0) * r_e_cm**2
mec2_keV     = 511.0                   # keV
DELTA_E_KEV  = 1.0                     # keV

# -- Grupos de energia (E_sup, E_inf] en keV
ENERGY_GROUPS = [
    (100, 90), (90, 80), (80, 70), (70, 60), (60, 50),
    (50, 40),  (40, 30), (30, 20), (20, 10), (10,  0),
]
N_GROUPS = len(ENERGY_GROUPS)

E_MIN_KEV = 20.0
E_MAX_KEV = 80.0


def active_groups() -> list:
    """Indices (0-based) de los grupos que solapan [E_MIN_KEV, E_MAX_KEV]."""
    return [g for g, (E_sup, E_inf) in enumerate(ENERGY_GROUPS)
            if E_sup > E_MIN_KEV and E_inf < E_MAX_KEV]


def energy_bins_in_group(g: int) -> np.ndarray:
    """Centros de bin (Delta_E=1 keV) de todos los bins del grupo g, sin recorte."""
    E_sup, E_inf = ENERGY_GROUPS[g]
    return np.arange(float(E_inf) + 0.5, float(E_sup) + 0.5, DELTA_E_KEV)


def energy_bins_in_range(g: int) -> list:
    """Centros de bin del grupo g recortados a [E_MIN_KEV, E_MAX_KEV]."""
    return [E for E in energy_bins_in_group(g) if E_MIN_KEV <= E <= E_MAX_KEV]


def group_of(E_keV: float) -> int:
    """Indice (0-based) del grupo de E_keV; devuelve -1 fuera de rango."""
    for g, (E_sup, E_inf) in enumerate(ENERGY_GROUPS):
        if E_inf < E_keV <= E_sup:
            return g
    return -1


# -- 26 Ordenadas discretas S26
#   m=0..5   cara       omega = 4*pi/21
#   m=6..17  arista     omega = 16*pi/105
#   m=18..25 vertice    omega = 9*pi/70
def _build_discrete_ordinates():
    """Construye las 26 direcciones S26 y pesos omega_m [sr].
    Ref: Acuros Sec.2.B; Vassiliev Sec.7.2.
    """
    s2, s3 = math.sqrt(2), math.sqrt(3)
    Omega_np = np.array([
        [ 1.,  0.,  0.], [-1.,  0.,  0.],
        [ 0.,  1.,  0.], [ 0., -1.,  0.],
        [ 0.,  0.,  1.], [ 0.,  0., -1.],
        [ 1./s2,  1./s2,  0.   ], [-1./s2, -1./s2,  0.   ],
        [ 1./s2, -1./s2,  0.   ], [-1./s2,  1./s2,  0.   ],
        [ 1./s2,  0.,     1./s2], [-1./s2,  0.,    -1./s2],
        [ 1./s2,  0.,    -1./s2], [-1./s2,  0.,     1./s2],
        [ 0.,     1./s2,  1./s2], [ 0.,    -1./s2, -1./s2],
        [ 0.,     1./s2, -1./s2], [ 0.,    -1./s2,  1./s2],
        [ 1./s3,  1./s3,  1./s3], [-1./s3, -1./s3, -1./s3],
        [-1./s3,  1./s3,  1./s3], [ 1./s3, -1./s3,  1./s3],
        [ 1./s3,  1./s3, -1./s3], [-1./s3, -1./s3,  1./s3],
        [ 1./s3, -1./s3, -1./s3], [-1./s3,  1./s3, -1./s3],
    ], dtype=float)
    omega_m_np = np.array(
        [4.0*math.pi/21.0]   * 6 +
        [16.0*math.pi/105.0] * 12 +
        [9.0*math.pi/70.0]   * 8, dtype=float)
    return Omega_np, omega_m_np

OMEGA_M, OMEGA_WEIGHTS = _build_discrete_ordinates()
N_DIR = 26

print(f'OK Celda 1 | Grupos activos: {active_groups()} | N_DIR={N_DIR} | sum_omega={OMEGA_WEIGHTS.sum():.4f}')


OK Celda 1 | Grupos activos: [2, 3, 4, 5, 6, 7] | N_DIR=26 | sum_omega=12.5664


## Celda 2 --- PointCrossSections

Secciones eficaces puntuales: fotoelectrico Biggs-Lighthill + Compton KN*S(x,Z).

Refs: Acuros Eq.(1); Vassiliev Sec.3.1; NJOY GAMINR Sec.9.1, Eq.(380).


In [3]:
# CELDA 2 -- PointCrossSections  [Acuros Eq.(1); NJOY GAMINR Sec.9.1]

import numpy as np
import math

_Z_ELEM = {'H': 1, 'O': 8, 'N': 7}

# -- Tablas S(x,Z) ENDF/B-VII.1 File 27, MT=504
# S(x,Z): funcion de dispersion incoherente; S(x->inf,Z) = Z.
# Variable: x = 20.60744*q  [Ang^-1], NJOY GAMINR Eq.(382-383).

# H (Z=1)
_S_X_H_ENDF = np.array([
    0.000000e+00,1.000000e-07,1.000000e-06,1.000000e-05,1.000000e-04,1.000000e-03,
    5.000000e-03,1.000000e-02,1.500000e-02,2.000000e-02,2.500000e-02,3.000000e-02,
    3.750000e-02,4.000000e-02,4.750000e-02,5.000000e-02,5.875000e-02,6.625000e-02,
    7.000000e-02,7.875000e-02,8.000000e-02,8.625000e-02,9.000000e-02,9.750000e-02,
    1.000000e-01,1.062500e-01,1.156300e-01,1.250000e-01,1.359400e-01,1.453100e-01,
    1.500000e-01,1.609400e-01,1.703100e-01,1.750000e-01,1.875000e-01,2.000000e-01,
    2.125000e-01,2.218800e-01,2.289100e-01,2.359400e-01,2.429700e-01,2.500000e-01,
    2.625000e-01,2.718800e-01,2.789100e-01,2.906300e-01,2.929700e-01,3.000000e-01,
    3.179700e-01,3.250000e-01,3.330100e-01,3.497600e-01,3.625000e-01,3.677000e-01,
    3.892300e-01,4.000000e-01,4.250000e-01,4.437500e-01,4.500000e-01,4.718800e-01,
    5.000000e-01,5.250000e-01,5.625000e-01,6.000000e-01,6.500000e-01,7.000000e-01,
    7.500000e-01,8.000000e-01,8.750000e-01,9.000000e-01,1.000000e+00,1.125000e+00,
    1.250000e+00,1.437500e+00,1.500000e+00,1.750000e+00,2.000000e+00,2.500000e+00,
    3.000000e+00,3.500000e+00,4.000000e+00,5.000000e+00,6.000000e+00,7.000000e+00,
    8.000000e+00,1.000000e+01,1.500000e+01,2.000000e+01,5.000000e+01,8.000000e+01,
    1.000000e+02,1.000000e+03,1.000000e+06,1.000000e+09,
], dtype=float)
_S_VAL_H_ENDF = np.array([
    0.000000e+00,4.409700e-13,4.409700e-11,4.409700e-09,4.409700e-07,4.409700e-05,
    1.102425e-03,4.409700e-03,9.887600e-03,1.749400e-02,2.716600e-02,3.882600e-02,
    5.983670e-02,6.772900e-02,9.383850e-02,1.033100e-01,1.391020e-01,1.726290e-01,
    1.902300e-01,2.329580e-01,2.392320e-01,2.710060e-01,2.903800e-01,3.294640e-01,
    3.425600e-01,3.752390e-01,4.238310e-01,4.713000e-01,5.245410e-01,5.678810e-01,
    5.887300e-01,6.347230e-01,6.712280e-01,6.885000e-01,7.310720e-01,7.688400e-01,
    8.019520e-01,8.240260e-01,8.392010e-01,8.530840e-01,8.658780e-01,8.776800e-01,
    8.961610e-01,9.082310e-01,9.163850e-01,9.283980e-01,9.305910e-01,9.368600e-01,
    9.502770e-01,9.546510e-01,9.592280e-01,9.673380e-01,9.723280e-01,9.741820e-01,
    9.804080e-01,9.829800e-01,9.875650e-01,9.901100e-01,9.908580e-01,9.930070e-01,
    9.950200e-01,9.962740e-01,9.975580e-01,9.983700e-01,9.989660e-01,9.994100e-01,
    9.995910e-01,9.997600e-01,9.998670e-01,9.999000e-01,9.999500e-01,9.999710e-01,
    9.999900e-01,9.999980e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,
    1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,
    1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,
    1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,
], dtype=float)

# O (Z=8)
_S_X_O_ENDF = np.array([
    0.000000e+00,1.000000e-07,1.000000e-06,1.000000e-05,1.000000e-04,1.000000e-03,
    5.000000e-03,6.250000e-03,7.187500e-03,7.890600e-03,8.945300e-03,1.000000e-02,
    1.500000e-02,1.750000e-02,2.000000e-02,2.500000e-02,3.000000e-02,4.000000e-02,
    5.000000e-02,6.000000e-02,7.000000e-02,8.000000e-02,8.500000e-02,9.000000e-02,
    1.000000e-01,1.125000e-01,1.250000e-01,1.375000e-01,1.500000e-01,1.625000e-01,
    1.750000e-01,1.875000e-01,1.937500e-01,2.000000e-01,2.125000e-01,2.312500e-01,
    2.500000e-01,2.718800e-01,2.906300e-01,3.000000e-01,3.250000e-01,3.437500e-01,
    3.625000e-01,3.718800e-01,4.000000e-01,4.250000e-01,4.625000e-01,4.750000e-01,
    5.000000e-01,5.500000e-01,5.875000e-01,6.000000e-01,6.500000e-01,7.000000e-01,
    7.500000e-01,8.000000e-01,9.000000e-01,1.000000e+00,1.109400e+00,1.203100e+00,
    1.250000e+00,1.312500e+00,1.406300e+00,1.500000e+00,1.589800e+00,1.665000e+00,
    1.748800e+00,1.750000e+00,1.838500e+00,1.919300e+00,2.000000e+00,2.062500e+00,
    2.335900e+00,2.375000e+00,2.445300e+00,2.500000e+00,2.750000e+00,2.835900e+00,
    2.894500e+00,2.947300e+00,3.000000e+00,3.062500e+00,3.117200e+00,3.212900e+00,
    3.392300e+00,3.464100e+00,3.500000e+00,3.562500e+00,3.671900e+00,3.877000e+00,
    3.959000e+00,4.000000e+00,4.091800e+00,4.176000e+00,4.279000e+00,4.327100e+00,
    5.000000e+00,5.125000e+00,5.343800e+00,5.753900e+00,5.918000e+00,6.000000e+00,
    6.562500e+00,7.000000e+00,8.000000e+00,1.000000e+01,1.500000e+01,1.000000e+09,
], dtype=float)
_S_VAL_O_ENDF = np.array([
    0.000000e+00,1.100000e-12,1.100000e-10,1.100000e-08,1.100000e-06,1.100000e-04,
    2.750000e-03,4.296875e-03,5.682617e-03,6.848773e-03,8.802023e-03,1.100000e-02,
    2.530000e-02,3.441520e-02,4.480000e-02,6.980000e-02,1.001000e-01,1.761000e-01,
    2.710000e-01,3.841910e-01,5.137000e-01,6.568110e-01,7.328590e-01,8.118000e-01,
    9.770000e-01,1.194080e+00,1.419900e+00,1.651210e+00,1.885000e+00,2.118480e+00,
    2.349700e+00,2.576840e+00,2.688710e+00,2.799000e+00,3.014080e+00,3.322920e+00,
    3.613500e+00,3.927910e+00,4.176130e+00,4.293000e+00,4.580310e+00,4.774320e+00,
    4.951080e+00,5.033340e+00,5.257000e+00,5.429000e+00,5.647990e+00,5.711810e+00,
    5.828000e+00,6.020960e+00,6.139280e+00,6.175000e+00,6.301780e+00,6.411000e+00,
    6.507880e+00,6.596000e+00,6.755000e+00,6.901000e+00,7.047320e+00,7.161580e+00,
    7.215900e+00,7.283300e+00,7.377340e+00,7.462000e+00,7.533340e+00,7.586660e+00,
    7.639770e+00,7.640450e+00,7.689840e+00,7.729440e+00,7.764200e+00,7.787300e+00,
    7.867040e+00,7.875420e+00,7.890010e+00,7.899900e+00,7.934000e+00,7.943210e+00,
    7.948220e+00,7.952650e+00,7.957000e+00,7.960700e+00,7.963880e+00,7.969320e+00,
    7.976770e+00,7.979400e+00,7.980700e+00,7.982170e+00,7.984680e+00,7.989170e+00,
    7.990400e+00,7.991000e+00,7.991770e+00,7.992470e+00,7.993300e+00,7.993680e+00,
    7.997700e+00,7.997920e+00,7.998280e+00,7.998930e+00,7.999180e+00,7.999300e+00,
    7.999590e+00,7.999800e+00,8.000000e+00,8.000000e+00,8.000000e+00,8.000000e+00,
], dtype=float)

# N (Z=7)
_S_X_N_ENDF = np.array([
    0.000000e+00,1.000000e-07,1.000000e-06,1.000000e-05,1.000000e-04,1.000000e-03,
    5.000000e-03,6.718700e-03,7.890600e-03,9.296900e-03,1.000000e-02,1.125000e-02,
    1.500000e-02,2.000000e-02,2.500000e-02,3.000000e-02,4.000000e-02,5.000000e-02,
    5.875000e-02,6.000000e-02,7.000000e-02,8.000000e-02,9.000000e-02,1.000000e-01,
    1.109400e-01,1.125000e-01,1.203100e-01,1.250000e-01,1.375000e-01,1.500000e-01,
    1.625000e-01,1.750000e-01,1.875000e-01,1.937500e-01,2.000000e-01,2.125000e-01,
    2.312500e-01,2.500000e-01,2.718800e-01,2.750000e-01,2.906300e-01,3.000000e-01,
    3.250000e-01,3.437500e-01,3.718800e-01,4.000000e-01,4.437500e-01,4.812500e-01,
    5.000000e-01,5.250000e-01,5.625000e-01,5.750000e-01,6.000000e-01,6.500000e-01,
    7.000000e-01,8.000000e-01,8.750000e-01,9.000000e-01,9.750000e-01,1.000000e+00,
    1.085900e+00,1.144500e+00,1.214800e+00,1.250000e+00,1.312500e+00,1.406300e+00,
    1.500000e+00,1.617200e+00,1.712900e+00,1.750000e+00,1.784700e+00,1.838500e+00,
    1.919300e+00,2.000000e+00,2.062500e+00,2.308600e+00,2.375000e+00,2.377000e+00,
    2.459000e+00,2.500000e+00,2.750000e+00,2.835900e+00,2.894500e+00,2.947300e+00,
    3.000000e+00,3.062500e+00,3.117200e+00,3.212900e+00,3.392300e+00,3.464100e+00,
    3.500000e+00,3.562500e+00,3.671900e+00,4.000000e+00,4.125000e+00,4.234400e+00,
    4.425800e+00,5.000000e+00,5.250000e+00,5.718800e+00,6.000000e+00,6.625000e+00,
    6.875000e+00,7.000000e+00,7.250000e+00,7.718800e+00,7.906300e+00,8.000000e+00,
    8.125000e+00,8.359400e+00,8.654200e+00,8.900900e+00,9.529000e+00,1.000000e+01,
    1.045900e+01,1.088000e+01,1.139500e+01,1.205600e+01,1.247700e+01,1.373800e+01,
    1.500000e+01,1.718800e+01,1.906300e+01,2.000000e+01,2.187500e+01,2.363300e+01,
    2.692900e+01,2.981300e+01,5.000000e+01,5.750000e+01,8.000000e+01,1.000000e+02,
    1.270300e+02,1.557000e+02,1.855600e+02,2.162100e+02,2.652000e+02,3.326500e+02,
    3.945400e+02,4.702300e+02,5.673500e+02,6.755100e+02,8.174800e+02,1.000000e+03,
    1.171400e+03,1.365400e+03,1.706700e+03,2.048600e+03,2.521700e+03,3.357900e+03,
    4.681900e+03,6.333900e+03,1.036300e+04,1.589300e+04,3.032700e+04,7.267300e+04,
    2.695800e+05,1.000000e+06,5.623400e+06,5.424700e+07,1.000000e+09,
], dtype=float)
_S_VAL_N_ENDF = np.array([
    0.000000e+00,1.300000e-12,1.300000e-10,1.300000e-08,1.300000e-06,1.300000e-04,
    3.250000e-03,5.868321e-03,8.094004e-03,1.123621e-02,1.300000e-02,1.649010e-02,
    2.920000e-02,5.170000e-02,8.040000e-02,1.151000e-01,2.017000e-01,3.100000e-01,
    4.200330e-01,4.368180e-01,5.797000e-01,7.364600e-01,9.042000e-01,1.080000e+00,
    1.278750e+00,1.307440e+00,1.452170e+00,1.539700e+00,1.772470e+00,2.003000e+00,
    2.228330e+00,2.446800e+00,2.657050e+00,2.758520e+00,2.858000e+00,3.048640e+00,
    3.315380e+00,3.558600e+00,3.812940e+00,3.846890e+00,4.007360e+00,4.097000e+00,
    4.311800e+00,4.453020e+00,4.636790e+00,4.792000e+00,4.987420e+00,5.122470e+00,
    5.182000e+00,5.254070e+00,5.350560e+00,5.380580e+00,5.437000e+00,5.540240e+00,
    5.635000e+00,5.809000e+00,5.929580e+00,5.968000e+00,6.077840e+00,6.113000e+00,
    6.226560e+00,6.298480e+00,6.378060e+00,6.415700e+00,6.477240e+00,6.558950e+00,
    6.630000e+00,6.703240e+00,6.753430e+00,6.770310e+00,6.785630e+00,6.806880e+00,
    6.835130e+00,6.859900e+00,6.875510e+00,6.923390e+00,6.932290e+00,6.932540e+00,
    6.942620e+00,6.947000e+00,6.966450e+00,6.971560e+00,6.974260e+00,6.976650e+00,
    6.979000e+00,6.980790e+00,6.982330e+00,6.984950e+00,6.989520e+00,6.990710e+00,
    6.991300e+00,6.991920e+00,6.992990e+00,6.996000e+00,6.996430e+00,6.996790e+00,
    6.997410e+00,6.999100e+00,6.999290e+00,6.999620e+00,6.999800e+00,6.999860e+00,
    6.999890e+00,6.999900e+00,6.999930e+00,6.999970e+00,6.999990e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
    7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00,
], dtype=float)

_SF_TABLES = {
    'H': (_S_X_H_ENDF, _S_VAL_H_ENDF),
    'O': (_S_X_O_ENDF, _S_VAL_O_ENDF),
    'N': (_S_X_N_ENDF, _S_VAL_N_ENDF),
}


def _compute_x_incoherent(E_keV: float, cos_theta: float) -> float:
    """Variable x = q*20.60744 para la dispersion incoherente. NJOY GAMINR Eq.(383)."""
    k = E_keV / mec2_keV
    dmu = 1.0 - cos_theta
    denom = 1.0 + k * dmu
    if denom <= 0.0:
        return 0.0
    num_sq = 1.0 + (k**2 + 2.0*k) * dmu / 2.0
    q = 2.0*k * math.sqrt(max(0.0, dmu/2.0)) * math.sqrt(max(0.0, num_sq)) / denom
    return 20.60744 * q


def _interp_scatter_function(x: float, elem: str) -> float:
    """Interpolacion log-log de S(x,Z) con tablas ENDF/B-VII.1 MT=504."""
    x_tbl, s_tbl = _SF_TABLES.get(elem, (None, None))
    if x_tbl is None or x <= 0.0:
        return 0.0
    if x >= x_tbl[-1]:
        return float(s_tbl[-1])
    eps = 1e-12
    mask = x_tbl > 0
    ln_x_tbl = np.log(x_tbl[mask])
    ln_s_tbl = np.log(np.clip(s_tbl[mask], eps, None))
    return float(math.exp(np.interp(math.log(x), ln_x_tbl, ln_s_tbl)))


class PointCrossSections:
    """Secciones eficaces puntuales: fotoelectrico Biggs-Lighthill + Compton KN*S(x,Z).

    API publica:
      photoelectric_mu_over_rho, compton_mu_over_rho, total_mu,
      macroscopic_total/compton/photoelectric,
      compton_incoherent_dsigma_dOmega_per_atom/per_mass/macroscopic,
      klein_nishina_dsigma_dOmega, compton_E_out, electron_density.

    Refs: Acuros Eq.(1); Vassiliev Sec.3.1; NJOY GAMINR Sec.9.1.
    """

    def __init__(self):
        self._pe_params = {
            'H': {
                'START':  [0.8,   4.0,   20.0,  100.0, 500.0],
                'FINISH': [4.0,   20.0,  100.0, 500.0, float('inf')],
                'A_1': [ 7.636E-02, 1.180E-03,+3.7783E-05, 1.034E-06, 4.599E-07],
                'A_2': [-9.406E-01,-8.236E-02,-9.2692E-03,-4.114E-04, 5.006E-04],
                'A_3': [ 6.144E+00, 2.886E+00,+1.3761E+00, 6.287E-01,-1.425E-02],
                'A_4': [ 1.425E+00, 5.534E+00,+1.5914E+01, 3.927E+01, 1.960E+02],
            },
            'N': {
                'START':  [0.8,   4.0,   20.0,  100.0, 500.0],
                'FINISH': [4.0,   20.0,  100.0, 500.0, float('inf')],
                'A_1': [-4.940E+00, 2.019E+00,+6.9367E-02, 1.872E-03, 8.122E-04],
                'A_2': [-8.442E+01,-1.249E+02,-1.7360E+01,-6.732E-01, 8.364E-01],
                'A_3': [ 4.620E+03, 4.609E+03,+2.8059E+03, 1.282E+03, 4.410E+02],
                'A_4': [-1.186E+03,-9.421E+02,+1.1254E+04, 5.700E+04, 2.358E+05],
            },
            'O': {
                'START':  [0.532, 4.0,   20.0,  100.0, 500.0],
                'FINISH': [4.0,   20.0,  100.0, 500.0, float('inf')],
                'A_1': [-7.181E+01, 2.745E+00,+1.1264E-01, 3.169E-03, 1.367E-03],
                'A_2': [ 4.748E+02,-1.747E+02,-2.8287E+01,-1.146E+00, 1.473E+00],
                'A_3': [ 5.542E+03, 7.159E+03,+4.6739E+03, 2.194E+03, 7.214E+02],
                'A_4': [-1.363E+03,-2.213E+03,+1.5005E+04, 9.131E+04, 4.048E+05],
            },
        }
        self._elements = ['H', 'O', 'N']
        self._Z_A    = {'H': 0.9921, 'O': 0.5, 'N': 4.998E-01}
        self._Z_ELEM = dict(_Z_ELEM)
        self._n_quad = 64
        self._mu_quad, self._w_quad = np.polynomial.legendre.leggauss(self._n_quad)

    # Klein-Nishina

    def _kn_dsigma_bare(self, E_in_keV: float, cos_theta: float) -> float:
        """dsigma_KN/dOmega [cm2/sr/e-] pura. Vassiliev Eq.(2.33)."""
        alpha = E_in_keV / mec2_keV
        if alpha <= 0:
            return 0.0
        k = 1.0 / (1.0 + alpha*(1.0 - cos_theta))
        if k <= 0:
            return 0.0
        return max(0.0, 0.5*r_e_cm**2 * k**2 * (k + 1.0/k - (1.0 - cos_theta**2)))

    def klein_nishina_dsigma_dOmega(self, E_in_keV: float, cos_theta: float) -> float:
        """dsigma_KN/dOmega [cm2/sr/e-]. Alias publico."""
        return self._kn_dsigma_bare(E_in_keV, cos_theta)

    # Seccion diferencial Compton

    def compton_incoherent_dsigma_dOmega_per_atom(self, E_in_keV, cos_theta, elem):
        """dsigma_C/dOmega [cm2/sr/atomo] = S(x,Z)*dsigma_KN. NJOY GAMINR Eq.(380)."""
        x_inc  = _compute_x_incoherent(E_in_keV, cos_theta)
        return max(0.0, _interp_scatter_function(x_inc, elem)
                        * self._kn_dsigma_bare(E_in_keV, cos_theta))

    def compton_incoherent_dsigma_dOmega_per_mass(self, E_in_keV, cos_theta, composition):
        """dsigma_C/dOmega [cm2/sr/g] para la mezcla. Vassiliev Sec.3.1."""
        dsig = self._kn_dsigma_bare(E_in_keV, cos_theta)
        if dsig <= 0.0:
            return 0.0
        x_inc = _compute_x_incoherent(E_in_keV, cos_theta)
        S_mix = 0.0
        for elem in self._elements:
            w_e = float(composition[elem].values[0]) / 100.0
            if w_e <= 0:
                continue
            S_mix += (w_e * self._Z_A[elem]
                      * _interp_scatter_function(x_inc, elem)
                      / float(self._Z_ELEM[elem]))
        return max(0.0, S_mix * N_A * dsig)

    def compton_incoherent_dsigma_dOmega_macroscopic(self, E_in_keV, cos_theta, rho, composition):
        """dSigma_C/dOmega [cm^-1/sr] = rho * dsigma_C/dOmega_masa."""
        return rho * self.compton_incoherent_dsigma_dOmega_per_mass(
            E_in_keV, cos_theta, composition)

    # Secciones integradas

    def photoelectric_mu_over_rho(self, E_keV: float, composition) -> float:
        """mu_PE/rho [cm2/g] por Biggs-Lighthill. Acuros Eq.(1)."""
        mu_pe = 0.0
        for elem in self._elements:
            w_e = float(composition[elem].values[0]) / 100.0
            if w_e <= 0:
                continue
            params = self._pe_params.get(elem)
            if params is None:
                continue
            for i in range(len(params['START'])):
                if params['START'][i] <= E_keV < params['FINISH'][i]:
                    A1, A2 = params['A_1'][i], params['A_2'][i]
                    A3, A4 = params['A_3'][i], params['A_4'][i]
                    mu_pe += w_e * (A1/E_keV + A2/E_keV**2
                                    + A3/E_keV**3 + A4/E_keV**4)
                    break
        return max(0.0, mu_pe)

    def compton_mu_over_rho(self, E_keV: float, composition) -> float:
        """mu_C/rho [cm2/g] = N_A*2*pi*integral(S*dsigma_KN dmu). NJOY GAMINR Eq.(380)."""
        if E_keV <= 0:
            return 0.0
        sigma_I = 0.0
        for mu_val, w_val in zip(self._mu_quad, self._w_quad):
            mu_f = float(mu_val)
            dsig_kn = self._kn_dsigma_bare(E_keV, mu_f)
            if dsig_kn <= 0.0:
                continue
            x_inc = _compute_x_incoherent(E_keV, mu_f)
            S_mix = 0.0
            for elem in self._elements:
                w_e = float(composition[elem].values[0]) / 100.0
                if w_e <= 0:
                    continue
                S_mix += (w_e * self._Z_A[elem]
                          * _interp_scatter_function(x_inc, elem)
                          / float(self._Z_ELEM[elem]))
            sigma_I += float(w_val) * dsig_kn * S_mix
        return max(0.0, sigma_I * 2.0*math.pi * N_A)

    def total_mu(self, E_keV: float, rho: float, composition) -> float:
        """Sigma_t [cm^-1] = rho*(mu_PE/rho + mu_C/rho). Acuros Eq.(1)."""
        return max(0.0, (self.photoelectric_mu_over_rho(E_keV, composition)
                         + self.compton_mu_over_rho(E_keV, composition)) * rho)

    def macroscopic_photoelectric(self, E_keV, rho, composition):
        """Sigma_PE [cm^-1]."""
        return rho * self.photoelectric_mu_over_rho(E_keV, composition)

    def macroscopic_compton(self, E_keV, rho, composition):
        """Sigma_C [cm^-1]."""
        return rho * self.compton_mu_over_rho(E_keV, composition)

    def macroscopic_total(self, E_keV, rho, composition):
        """Sigma_t [cm^-1]. Alias de total_mu."""
        return self.total_mu(E_keV, rho, composition)

    def compton_E_out(self, E_in_keV: float, cos_theta: float) -> float:
        """E_out [keV] = E / [1 + (E/mec2)*(1-cos)]. Vassiliev Eq.(2.34)."""
        denom = 1.0 + (E_in_keV / mec2_keV) * (1.0 - cos_theta)
        return E_in_keV / denom if denom > 0 else 0.0

    def electron_density(self, rho: float, composition) -> float:
        """n_e [e^-/cm3] = rho*N_A*sum_i w_i*(Z/A)_i. Vassiliev Sec.3.1."""
        e_per_g = sum(
            float(composition[el].values[0]) / 100.0 * self._Z_A[el]
            for el in self._elements
            if float(composition[el].values[0]) > 0 and el in self._Z_A
        )
        return rho * N_A * e_per_g


print('OK PointCrossSections definida')


OK PointCrossSections definida


## Celda 3 --- UncollidedFluence

Fluencia sin colision desde fuente puntual con cuadratura S26.

  psi_UC(r_i, E_n, m') = phi_E(E_n,d) / omega_{m'} * exp(-RPL)

Refs: Acuros Eq.(3), Eq.(15); Vassiliev Eq.(7.4).


In [4]:
# CELDA 3 -- UncollidedFluence  [Acuros Eq.(3), Eq.(15); Vassiliev Eq.(7.4)]


class UncollidedFluence:
    """Fluencia sin colision desde fuente puntual con cuadratura S26.

    Args:
        point_xs   : PointCrossSections
        siddon_fn  : callable(source, target, E_keV) -> RPL [adim]
        spectrum_fn: callable(E_keV, dist_cm) -> phi_E [fotones/(cm2*keV)]
        geometry   : dict con source_cm, Omega (26,3), omega_m (26,)

    Refs: Acuros Eq.(3), Eq.(15); Vassiliev Eq.(7.4).
    """

    def __init__(self, point_xs, siddon_fn, spectrum_fn, geometry: dict):
        self.xs       = point_xs
        self.siddon   = siddon_fn
        self.spectrum = spectrum_fn
        self.source   = np.asarray(geometry['source_cm'], dtype=float)
        self.Omega    = np.asarray(geometry['Omega'],   dtype=float)
        self.omega_m  = np.asarray(geometry['omega_m'], dtype=float)
        self._cache: dict = {}

    def _primary_direction(self, r_i: np.ndarray):
        """m', Omega_UC y distancia d para la ordenada mas alineada con fuente->r_i.
        Acuros Sec.2.B.
        """
        d_vec = r_i - self.source
        d = float(np.linalg.norm(d_vec))
        if d < 1e-12:
            return None, None, None
        Omega_UC = d_vec / d
        m_prim   = int(np.argmax(self.Omega @ Omega_UC))
        return m_prim, Omega_UC, d

    def bin_to_bin(self, r_i: np.ndarray, E_keV: float, m_prime: int) -> float:
        """psi_UC(r_i, E_n, m') [fotones/(cm2*sr*keV)].
        = phi_E(E_n,d)/omega_{m'} * exp(-RPL).
        Acuros Eq.(3), Eq.(15).
        """
        r_i = np.asarray(r_i, dtype=float)
        m_prim, _, d = self._primary_direction(r_i)
        if m_prim is None or m_prime != m_prim:
            return 0.0
        cache_key = (tuple(np.round(r_i, 6)), float(E_keV), int(m_prime), 'bin')
        if cache_key in self._cache:
            return self._cache[cache_key]
        phi_E = float(self.spectrum(E_keV, d))
        om    = float(self.omega_m[m_prime])
        if om <= 0:
            return 0.0
        rpl    = self.siddon(self.source, r_i, E_keV)
        result = (phi_E / om) * math.exp(-rpl)
        self._cache[cache_key] = result
        return result

    def multigroup(self, r_i: np.ndarray, g: int, m_prime: int) -> float:
        """psi_UC_{i,g,m'} [fotones/(cm2*sr)] = Delta_E * sum_{n in g} psi_UC_bin.
        Solo bins en [E_MIN_KEV, E_MAX_KEV]. Vassiliev Eq.(7.4).
        """
        r_i = np.asarray(r_i, dtype=float)
        cache_key = (tuple(np.round(r_i, 6)), int(g), int(m_prime), 'mg')
        if cache_key in self._cache:
            return self._cache[cache_key]
        total = sum(
            DELTA_E_KEV * self.bin_to_bin(r_i, float(E_n), m_prime)
            for E_n in energy_bins_in_range(g)
        )
        self._cache[cache_key] = total
        return total


print('OK UncollidedFluence definida')


OK UncollidedFluence definida


## Celda 4 --- MultiGroupCrossSections

Colapso multigrupo de Sigma_t y del kernel Sigma_s, ponderados con psi_UC.

Refs: Vassiliev Sec.7.2.1; Acuros Eq.(16).


In [5]:
# CELDA 4 -- MultiGroupCrossSections  [Vassiliev Sec.7.2.1; Acuros Eq.(16)]


class MultiGroupCrossSections:
    """Colapso multigrupo de Sigma_t y del kernel Sigma_s^{g'->g}(m'->m).

    Args:
        point_xs   : PointCrossSections
        uc_fluence : UncollidedFluence
        material_fn: callable(r_i) -> (rho, composition, organ_id)

    Refs: Vassiliev Sec.7.2.1; Acuros Eq.(16).
    """

    def __init__(self, point_xs, uc_fluence, material_fn):
        self.xs       = point_xs
        self.uc       = uc_fluence
        self.material = material_fn

    def sigma_total_multigroup(self, r_i: np.ndarray, g: int, m_prime: int) -> float:
        """Sigma_{t,g}(r_i) [cm^-1] ponderado con psi_UC. Vassiliev Sec.7.2.1.
        Fallback al valor puntual en el centro del grupo si psi_UC ~ 0.
        """
        r_i = np.asarray(r_i, dtype=float)
        rho, comp, _ = self.material(r_i)
        num, den = 0.0, 0.0
        for E_n in energy_bins_in_range(g):
            psi = self.uc.bin_to_bin(r_i, float(E_n), m_prime)
            sig = self.xs.macroscopic_total(float(E_n), rho, comp)
            num += DELTA_E_KEV * sig * psi
            den += DELTA_E_KEV * psi
        if den <= 0.0:
            E_sup, E_inf = ENERGY_GROUPS[g]
            return self.xs.macroscopic_total((E_sup + E_inf) / 2.0, rho, comp)
        return num / den

    def sigma_total(self, r_i, g, m_prime):
        """Alias de sigma_total_multigroup."""
        return self.sigma_total_multigroup(r_i, g, m_prime)

    def sigma_scatter_material_multigroup(self, r_i: np.ndarray,
                                          g_prime: int, g: int,
                                          m_prime: int, m: int) -> float:
        """Sigma_s^{g'->g}(m'->m) [cm^-1/sr] para la mezcla. Acuros Eq.(16)."""
        r_i = np.asarray(r_i, dtype=float)
        rho, comp, _ = self.material(r_i)
        E_sup_out, E_inf_out = ENERGY_GROUPS[g]
        cos_theta = float(np.dot(self.uc.Omega[m_prime], self.uc.Omega[m]))
        num, den = 0.0, 0.0
        for E_prime in energy_bins_in_range(g_prime):
            psi_uc = self.uc.bin_to_bin(r_i, float(E_prime), m_prime)
            den += DELTA_E_KEV * psi_uc
            E_out = self.xs.compton_E_out(float(E_prime), cos_theta)
            if E_inf_out < E_out <= E_sup_out:
                dSig = self.xs.compton_incoherent_dsigma_dOmega_macroscopic(
                    float(E_prime), cos_theta, rho, comp)
                num += DELTA_E_KEV * dSig * psi_uc
        return num / den if den > 0.0 else 0.0

    def sigma_scatter(self, r_i, g_prime, g, m_prime, m):
        """Alias de sigma_scatter_material_multigroup."""
        return self.sigma_scatter_material_multigroup(r_i, g_prime, g, m_prime, m)

    def sigma_scatter_per_element_multigroup(self, r_i: np.ndarray,
                                             g_prime: int, g: int,
                                             mu, elem: str) -> float:
        """Sigma_s,elem^{g'->g} [cm^-1/sr] para un unico elemento.
        Util para verificar aditividad de Bragg.
        """
        r_i = np.asarray(r_i, dtype=float)
        rho, comp, _ = self.material(r_i)
        if elem not in self.xs._elements:
            raise ValueError(f'Elemento no soportado: {elem}')
        w_e = float(comp[elem].values[0]) / 100.0
        if w_e <= 0.0:
            return 0.0
        A_e = self.xs._Z_ELEM[elem] / self.xs._Z_A[elem]
        E_sup_out, E_inf_out = ENERGY_GROUPS[g]
        num, den = 0.0, 0.0
        for E_prime in energy_bins_in_range(g_prime):
            psi_uc = self.uc.bin_to_bin(r_i, float(E_prime), mu)
            den += DELTA_E_KEV * psi_uc
            E_out = self.xs.compton_E_out(float(E_prime), mu)
            if E_inf_out < E_out <= E_sup_out:
                ds_atom   = self.xs.compton_incoherent_dsigma_dOmega_per_atom(
                    float(E_prime), mu, elem)
                dSig_elem = rho * N_A * (w_e / A_e) * ds_atom
                num += DELTA_E_KEV * dSig_elem * psi_uc
        return num / den if den > 0.0 else 0.0

    def sigma_scatter_material_multigroup_at_cosine(self, r_i: np.ndarray,
                                                    g_prime: int, g: int,
                                                    mu: float,
                                                    weighting: str = 'flat') -> float:
        """Sigma_s^{g'->g}(mu) [cm^-1/sr] para mu continuo arbitrario.
        weighting='flat' equivale a NJOY iwt=2. Vassiliev Eq.(7.7).
        """
        r_i = np.asarray(r_i, dtype=float)
        rho, comp, _ = self.material(r_i)
        E_sup_out, E_inf_out = ENERGY_GROUPS[g]
        num, den = 0.0, 0.0
        for E_prime in energy_bins_in_range(g_prime):
            w = 1.0 if weighting == 'flat' else 1.0 / float(E_prime)
            den += DELTA_E_KEV * w
            E_out = self.xs.compton_E_out(float(E_prime), float(mu))
            if E_inf_out < E_out <= E_sup_out:
                dSig = self.xs.compton_incoherent_dsigma_dOmega_macroscopic(
                    float(E_prime), float(mu), rho, comp)
                num += DELTA_E_KEV * dSig * w
        return num / den if den > 0.0 else 0.0


print('OK MultiGroupCrossSections definida')


OK MultiGroupCrossSections definida


## Celda 5 --- FirstCollisionSource

s^FC_{i,g,m} = sum_{g'} omega_{m'} * Sigma_s^{g'->g}(m'->m) * psi_UC_{i,g',m'}

Refs: Acuros Eq.(4), Eq.(16).


In [6]:
# CELDA 5 -- FirstCollisionSource  [Acuros Eq.(4), Eq.(16)]


class FirstCollisionSource:
    """Fuente de primera colision multigrupo con cuadratura S26.

    s^FC_{i,g,m} [cm^-3/sr] = sum_{g'} omega_{m'} * Sigma_s^{g'->g}(m'->m) * psi_UC_{i,g',m'}

    Refs: Acuros Eq.(4), Eq.(16).
    """

    def __init__(self, mg_xs, uc_fluence):
        self.mg_xs = mg_xs
        self.uc    = uc_fluence

    def compute(self, r_i: np.ndarray, g: int, m: int) -> float:
        """s^FC_{i,g,m} [cm^-3/sr]. Suma sobre grupos activos usando m' primaria."""
        r_i = np.asarray(r_i, dtype=float)
        m_prim, _, _ = self.uc._primary_direction(r_i)
        if m_prim is None:
            return 0.0
        total = 0.0
        for g_prime in active_groups():
            psi_uc = self.uc.multigroup(r_i, g_prime, m_prim)
            if psi_uc <= 0.0:
                continue
            sig_s = self.mg_xs.sigma_scatter(r_i, g_prime, g, m_prim, m)
            total += float(self.uc.omega_m[m_prim]) * sig_s * psi_uc
        return total


print('OK FirstCollisionSource definida')


OK FirstCollisionSource definida


## Celda 6 --- BTE (clase principal)

Fachada del solver. Centraliza geometria, materiales, espectro SpekPy y ray tracing de Siddon.
Construye internamente xs / uc / mg_xs / fc.

Refs: Acuros Sec.2; Vassiliev Sec.7.2.


In [7]:
# CELDA 6 -- BTE  [Acuros Sec.2; Vassiliev Sec.7.2]


class BTE:
    """Fachada del solver LBTE para fotones kV.

    Orquesta geometria, espectro SpekPy, ray tracing de Siddon y la jerarquia
    de clases xs / uc / mg_xs / fc.

    API publica principal:
      fluencia_sin_colisionar(r_i, E_keV, m')        -> psi_UC bin*bin
      fluencia_sin_colisionar_grupo(r_i, g, m')       -> psi_UC multigrupo
      sigma_total_multigroup(r_i, g, m')              -> Sigma_t [cm^-1]
      sigma_scatter_multigroup(r_i, g', g, m', m)     -> Sigma_s [cm^-1/sr]
      primera_fuente_dispersion(r_i, g, m)            -> s^FC [cm^-3/sr]
      get_local_material(r_i)                         -> (rho, comp, organ_id)
      get_radiological_path_length(target, E_keV)     -> RPL [adim]

    Refs: Acuros Sec.2; Vassiliev Sec.7.2.
    """

    def __init__(self, xp, _CUPY_OK, siddonraytracer, sp_module):
        self.xp              = xp
        self._CUPY_OK        = _CUPY_OK
        self.siddonraytracer = siddonraytracer
        self.sp              = sp_module

        # -- Geometria
        self.source_phantom_distance_cm  = 100.0
        self.voxel_size_x_cm             = 1.0
        self.voxel_size_y_cm             = 1.0
        self.voxel_size_z_cm             = 1.0
        self.water_cube_voxels_x         = 30
        self.water_cube_voxels_y         = 15
        self.water_cube_voxels_z         = 30
        self.air_margin_sides_cm         = 15
        self.air_margin_back_cm          = 30
        self.air_margin_front_cm         = 1

        nx = self.water_cube_voxels_x + 2*int(self.air_margin_sides_cm)
        ny = int(self.air_margin_front_cm + self.source_phantom_distance_cm
                 + self.water_cube_voxels_y * self.voxel_size_y_cm
                 + self.air_margin_back_cm)
        nz = self.water_cube_voxels_z + 2*int(self.air_margin_sides_cm)

        self.world_number_of_voxels_x = nx
        self.world_number_of_voxels_y = ny
        self.world_number_of_voxels_z = nz

        # Mapa de voxeles: 0=aire, 1=agua
        self.world = self.xp.zeros((nx, ny, nz), dtype=self.xp.int32)
        si = int(self.air_margin_sides_cm)
        sj = int(self.air_margin_front_cm + self.source_phantom_distance_cm)
        sk = int(self.air_margin_sides_cm)
        self._water_start_world = (si, sj, sk)
        self.world[si:si + self.water_cube_voxels_x,
                   sj:sj + self.water_cube_voxels_y,
                   sk:sk + self.water_cube_voxels_z] = 1

        self.source_position_cm = np.array([0.0, 0.0, 0.0], dtype=float)
        self.world_start_cm = (
            -(nx * self.voxel_size_x_cm) / 2.0,
            -float(self.air_margin_front_cm),
            -(nz * self.voxel_size_z_cm) / 2.0)
        self.world_spacing_cm = (self.voxel_size_x_cm,
                                  self.voxel_size_y_cm,
                                  self.voxel_size_z_cm)

        # -- Materiales
        self.media = pd.DataFrame({
            'Media_number': [0, 1],
            'Media_name':   ['Air', 'Water'],
            'Density':      [0.0012, 1.0],
            'H':  [ 0.0,     11.1898],
            'O':  [22.0,     88.8102],
            'N':  [78.0,      0.0],
        })
        self._media_by_number = self.media.set_index('Media_number')
        self._organ_density   = {0: 0.0012, 1: 1.0}
        self._organ_tissue    = {0: 0, 1: 1}
        self._max_organ_id    = 1
        self._spectrum_cache  = {}
        self._mu_cache        = {}

        # -- Closures para las subclases
        def _siddon_fn(source, target, E_keV):
            return self.get_radiological_path_length(list(target), float(E_keV))

        def _spectrum_fn(E_keV, dist_cm):
            energies, fluences = self._get_spectrum_cached(dist_cm)
            return float(np.interp(float(E_keV), energies, fluences))

        def _material_fn(r_i):
            return self.get_local_material(np.asarray(r_i, dtype=float))

        # -- Cuadratura S26
        self.Omega   = OMEGA_M
        self.omega_m = OMEGA_WEIGHTS
        self.n_dir   = N_DIR

        geometry = {'source_cm': self.source_position_cm,
                    'Omega':     self.Omega,
                    'omega_m':   self.omega_m}

        # -- Jerarquia de clases
        self.xs    = PointCrossSections()
        self.uc    = UncollidedFluence(self.xs, _siddon_fn, _spectrum_fn, geometry)
        self.mg_xs = MultiGroupCrossSections(self.xs, self.uc, _material_fn)
        self.fc    = FirstCollisionSource(self.mg_xs, self.uc)

    # -- Utilidades GPU/CPU

    def _to_cpu(self, arr):
        return arr.get() if self._CUPY_OK and hasattr(arr, 'get') else np.asarray(arr)

    def _to_gpu(self, arr, dtype=None):
        if self._CUPY_OK:
            import cupy as cp
            return cp.asarray(arr, dtype=dtype) if dtype else cp.asarray(arr)
        return np.asarray(arr, dtype=dtype) if dtype else np.asarray(arr)

    def _clamp_point_to_world(self, point_cm):
        eps = 1e-3
        x0, y0, z0 = self.world_start_cm
        x1 = x0 + self.world_number_of_voxels_x * self.voxel_size_x_cm
        y1 = y0 + self.world_number_of_voxels_y * self.voxel_size_y_cm
        z1 = z0 + self.world_number_of_voxels_z * self.voxel_size_z_cm
        return np.array([np.clip(point_cm[0], x0+eps, x1-eps),
                         np.clip(point_cm[1], y0+eps, y1-eps),
                         np.clip(point_cm[2], z0+eps, z1-eps)], dtype=float)

    # -- Geometria

    def voxel_center_cm(self, i, j, k):
        """Coordenadas [cm] del centro del voxel (i,j,k)."""
        return np.array([
            self.world_start_cm[0] + (i + 0.5) * self.voxel_size_x_cm,
            self.world_start_cm[1] + (j + 0.5) * self.voxel_size_y_cm,
            self.world_start_cm[2] + (k + 0.5) * self.voxel_size_z_cm], dtype=float)

    def water_voxel_to_world_voxel(self, i_w, j_w, k_w):
        """Indices locales del maniqui de agua -> indices globales del mundo."""
        return (self._water_start_world[0] + i_w,
                self._water_start_world[1] + j_w,
                self._water_start_world[2] + k_w)

    # -- Espectro

    def _get_spectrum_cached(self, dist_cm):
        """Espectro SpekPy (100 kVp, W, 12 deg, Al 3 mm) escalado 1/r2. Cacheado."""
        key = float(dist_cm)
        if key not in self._spectrum_cache:
            s = self.sp.Spek(kvp=100.0, th=12.0, targ='W', dk=1.0, mas=1.0)
            s.filter('Al', 3.0)
            energies, fluences_100 = s.get_spectrum()
            fluences = np.asarray(fluences_100, dtype=np.float64) * (100.0/dist_cm)**2
            self._spectrum_cache[key] = (np.asarray(energies, dtype=np.float64), fluences)
        return self._spectrum_cache[key]

    # -- Ray tracing / RPL

    def _build_mu_lookup(self, E_keV):
        """Vector mu_t [cm^-1] indexado por organ_id para la energia E_keV."""
        mu_lookup = np.zeros(self._max_organ_id + 1, dtype=np.float64)
        for organ_id, tissue in self._organ_tissue.items():
            density = float(self._organ_density.get(organ_id, 0.0))
            if density <= 0:
                continue
            comp = self._media_by_number.loc[tissue:tissue]
            mu_lookup[int(organ_id)] = max(0.0, self.xs.total_mu(E_keV, density, comp))
        return mu_lookup

    def _get_mu_lookup(self, E_keV):
        key = float(E_keV)
        if key not in self._mu_cache:
            self._mu_cache[key] = self._build_mu_lookup(key)
        return self._mu_cache[key]

    def get_local_material(self, r_i_cm):
        """(rho, composition, organ_id) del voxel que contiene r_i_cm."""
        i = int(np.clip(int((r_i_cm[0] - self.world_start_cm[0]) / self.voxel_size_x_cm),
                        0, self.world_number_of_voxels_x - 1))
        j = int(np.clip(int((r_i_cm[1] - self.world_start_cm[1]) / self.voxel_size_y_cm),
                        0, self.world_number_of_voxels_y - 1))
        k = int(np.clip(int((r_i_cm[2] - self.world_start_cm[2]) / self.voxel_size_z_cm),
                        0, self.world_number_of_voxels_z - 1))
        organ_id = int(self._to_cpu(self.world)[i, j, k])
        tissue   = self._organ_tissue[organ_id]
        rho      = float(self._organ_density[organ_id])
        comp     = self._media_by_number.loc[tissue:tissue]
        return rho, comp, organ_id

    def get_radiological_path_length(self, target_cm, energy_keV):
        """RPL [adim] = integral(mu_t dl) via Siddon GPU/CPU."""
        alphas, lengths, _, raylength, indices = self.siddonraytracer(
            [self._to_gpu(self.world)],
            self._clamp_point_to_world(self.source_position_cm),
            self._clamp_point_to_world(np.array(target_cm, dtype=float)),
            np.array(self.world_start_cm,   dtype=float),
            np.array(self.world_spacing_cm, dtype=float))
        if self._CUPY_OK:
            import cupy as cp
            lengths_seq = cp.asnumpy(lengths) if isinstance(lengths, cp.ndarray) else np.asarray(lengths)
            indices_arr = cp.asnumpy(indices) if isinstance(indices, cp.ndarray) else np.asarray(indices)
        else:
            lengths_seq, indices_arr = np.asarray(lengths), np.asarray(indices)
        if lengths_seq.size == 0:
            return 0.0
        world_cpu = self._to_cpu(self.world)
        organ_ids = np.array(
            [int(world_cpu[int(indices_arr[i,0]),
                           int(indices_arr[i,1]),
                           int(indices_arr[i,2])])
             for i in range(len(indices_arr))], dtype=np.int32)
        mu_lookup = self._get_mu_lookup(float(energy_keV))
        mu_line   = mu_lookup[np.clip(organ_ids, 0, self._max_organ_id)]
        return max(0.0, float(np.dot(lengths_seq, mu_line)))

    # -- API publica: fluencia y fuente

    def fluencia_sin_colisionar(self, r_i, E_keV, m_prime):
        """psi_UC bin*bin [fotones/(cm2*sr*keV)]. Acuros Eq.(3), Eq.(15)."""
        return self.uc.bin_to_bin(np.asarray(r_i), E_keV, m_prime)

    def fluencia_sin_colisionar_grupo(self, r_i, g, m_prime):
        """psi_UC multigrupo [fotones/(cm2*sr)]. Vassiliev Eq.(7.4)."""
        return self.uc.multigroup(np.asarray(r_i), g, m_prime)

    def primera_fuente_dispersion(self, r_i, g, m):
        """s^FC_{i,g,m} [cm^-3/sr]. Acuros Eq.(4)."""
        return self.fc.compute(np.asarray(r_i), g, m)

    def primary_direction(self, r_i):
        """Indice m' de la ordenada S26 mas alineada con fuente->r_i."""
        m_prim, _, _ = self.uc._primary_direction(np.asarray(r_i))
        return m_prim

    # -- API publica: secciones eficaces multigrupo

    def sigma_total_multigroup(self, r_i, g, m_prime):
        """Sigma_{t,g} [cm^-1]. Vassiliev Eq.(7.6)."""
        return self.mg_xs.sigma_total_multigroup(np.asarray(r_i), g, m_prime)

    def sigma_scatter_multigroup(self, r_i, g_prime, g, m_prime, m):
        """Sigma_s^{g'->g}(m'->m) [cm^-1/sr]. Acuros Eq.(16)."""
        return self.mg_xs.sigma_scatter_material_multigroup(
            np.asarray(r_i), g_prime, g, m_prime, m)

    def sigma_scatter_material_multigroup(self, r_i, g_prime, g, m_prime, m):
        return self.mg_xs.sigma_scatter_material_multigroup(
            np.asarray(r_i), g_prime, g, m_prime, m)

    def sigma_scatter_per_element_multigroup(self, r_i, g_prime, g, m_prime, m, elem):
        """Sigma_s,elem^{g'->g}(m'->m) [cm^-1/sr] por elemento. Vassiliev Sec.3.1."""
        return self.mg_xs.sigma_scatter_per_element_multigroup(
            np.asarray(r_i), g_prime, g, m_prime, m, elem)

    def sigma_scatter_material_multigroup_at_cosine(self, r_i, g_prime, g,
                                                     cos_theta, weighting='flat'):
        """Sigma_s^{g'->g}(mu) [cm^-1/sr] para mu continuo. Vassiliev Eq.(7.7)."""
        return self.mg_xs.sigma_scatter_material_multigroup_at_cosine(
            np.asarray(r_i), g_prime, g, cos_theta, weighting)

    # -- API publica: secciones eficaces puntuales

    def photoelectric_mu_over_rho(self, E_keV, composition):
        return self.xs.photoelectric_mu_over_rho(E_keV, composition)

    def compton_mu_over_rho(self, E_keV, composition):
        return self.xs.compton_mu_over_rho(E_keV, composition)

    def macroscopic_photoelectric(self, E_keV, rho, composition):
        return self.xs.macroscopic_photoelectric(E_keV, rho, composition)

    def macroscopic_compton(self, E_keV, rho, composition):
        return self.xs.macroscopic_compton(E_keV, rho, composition)

    def macroscopic_total(self, E_keV, rho, composition):
        return self.xs.macroscopic_total(E_keV, rho, composition)

    def klein_nishina_dsigma_dOmega(self, E_keV, cos_theta):
        return self.xs.klein_nishina_dsigma_dOmega(E_keV, cos_theta)

    def compton_incoherent_dsigma_dOmega_per_atom(self, E_keV, cos_theta, elem):
        return self.xs.compton_incoherent_dsigma_dOmega_per_atom(E_keV, cos_theta, elem)

    def compton_incoherent_dsigma_dOmega_per_mass(self, E_keV, cos_theta, composition):
        return self.xs.compton_incoherent_dsigma_dOmega_per_mass(E_keV, cos_theta, composition)

    def compton_incoherent_dsigma_dOmega_macroscopic(self, E_keV, cos_theta, rho, composition):
        return self.xs.compton_incoherent_dsigma_dOmega_macroscopic(
            E_keV, cos_theta, rho, composition)

    def compton_E_out(self, E_keV, cos_theta):
        return self.xs.compton_E_out(E_keV, cos_theta)


print('OK BTE definida')


OK BTE definida


## Celda 7 --- Instanciacion y ejemplo minimo de uso


In [8]:
# CELDA 7 -- Instanciacion y ejemplo minimo de uso

bte = BTE(xp, _CUPY_OK, siddonraytracer, sp)

print('OK BTE instanciado correctamente.')
print(f'   Mundo : {bte.world_number_of_voxels_x}x'
      f'{bte.world_number_of_voxels_y}x'
      f'{bte.world_number_of_voxels_z} voxeles')
print(f'   Maniqui agua : {bte.water_cube_voxels_x}x'
      f'{bte.water_cube_voxels_y}x'
      f'{bte.water_cube_voxels_z} cm3')
print(f'   Fuente       : {bte.source_position_cm} cm')
print(f'   Inicio mundo Y: {bte.world_start_cm[1]:.1f} cm')

# Ejemplo minimo: magnitudes en el centro del maniqui de agua
r_c   = np.array([0.0, 107.5, 0.0])
E_REF = 50.0
G_REF = active_groups()[len(active_groups())//2]
m_p   = bte.primary_direction(r_c)

print()
print(f"r_c={r_c} cm  E={E_REF} keV  g={G_REF}  m'={m_p}")
print('-'*60)

psi_bin = bte.fluencia_sin_colisionar(r_c, E_REF, m_p)
psi_g   = bte.fluencia_sin_colisionar_grupo(r_c, G_REF, m_p)
sig_g   = bte.sigma_total_multigroup(r_c, G_REF, m_p)
m_back  = (m_p + 1) % N_DIR
sig_s   = bte.sigma_scatter_multigroup(r_c, G_REF, G_REF, m_p, m_back)
sfc     = bte.primera_fuente_dispersion(r_c, G_REF, m_p)

print(f'  psi_UC bin*bin       = {psi_bin:.3e}  fotones/(cm2*sr*keV)')
print(f'  psi_UC grupo g={G_REF}   = {psi_g:.3e}  fotones/(cm2*sr)')
print(f'  Sigma_t grupo g={G_REF}  = {sig_g:.5f}  cm^-1')
print(f'  Sigma_s(g->g, back) = {sig_s:.3e}  cm^-1/sr')
print(f'  s^FC(g={G_REF}, m={m_p})   = {sfc:.3e}  cm^-3/sr')
print()

assert psi_bin >= 0.0, 'psi_UC bin negativa'
assert psi_g   >= 0.0, 'psi_UC grupo negativa'
assert sig_g   >  0.0, 'Sigma_t nula o negativa'
print('OK Sanity check superado -- solver listo.')


OK BTE instanciado correctamente.
   Mundo : 60x146x60 voxeles
   Maniqui agua : 30x15x30 cm3
   Fuente       : [0. 0. 0.] cm
   Inicio mundo Y: -1.0 cm

r_c=[  0.  107.5   0. ] cm  E=50.0 keV  g=5  m'=2
------------------------------------------------------------
  psi_UC bin*bin       = 1.147e+06  fotones/(cm2*sr*keV)
  psi_UC grupo g=5   = 1.135e+07  fotones/(cm2*sr)
  Sigma_t grupo g=5  = 0.22115  cm^-1
  Sigma_s(g->g, back) = 5.805e-03  cm^-1/sr
  s^FC(g=5, m=2)   = 0.000e+00  cm^-3/sr

OK Sanity check superado -- solver listo.
